# 03 — Test des Agents & Tools (P2)

Ce notebook teste :
1. Chaque **tool en isolation** (météo, recherche web, calculatrice)
2. L'**agent LangGraph** complet qui choisit automatiquement le bon outil

**Prérequis** : avoir complété l'installation et configuré `.env` avec `GROQ_API_KEY`.

In [2]:
import sys
sys.path.insert(0, '..')  # remonter à la racine du projet

from dotenv import load_dotenv
load_dotenv()
print('✅ Environnement chargé')

✅ Environnement chargé


---
## 1. Test isolé — `get_weather` (OpenMeteo)

In [3]:
from src.agents.tools import get_weather

# Test 1 : ville française
result = get_weather.invoke({'city': 'Paris'})
print(result)

🌍 Météo actuelle à Paris
• Conditions    : Couvert ☁️
• Température   : 18.1 °C (ressenti 16.3 °C)
• Humidité      : 66 %
• Vent          : 15.9 km/h
• Précipitations: 0.0 mm
• Pression      : 1013.1 hPa


In [4]:
# Test 2 : ville internationale
result = get_weather.invoke({'city': 'Dakar'})
print(result)

🌍 Météo actuelle à Dakar
• Conditions    : Partiellement nuageux ⛅
• Température   : 21.4 °C (ressenti 20.7 °C)
• Humidité      : 82 %
• Vent          : 25.6 km/h
• Précipitations: 0.0 mm
• Pression      : 1009.4 hPa


In [5]:
# Test 3 : ville introuvable (cas d'erreur)
result = get_weather.invoke({'city': 'lyon'})
print(result)

🌍 Météo actuelle à Lyon
• Conditions    : Ciel dégagé ☀️
• Température   : 21.3 °C (ressenti 19.4 °C)
• Humidité      : 38 %
• Vent          : 6.9 km/h
• Précipitations: 0.0 mm
• Pression      : 996.2 hPa


---
## 2. Test isolé — `web_search` (DuckDuckGo)

In [6]:
from src.agents.tools import web_search

# Test 1 : actualité climatique récente
result = web_search.invoke({'query': 'catastrophes climatiques 2024 bilan', 'max_results': 3})
print(result)

c:\Users\Camille\mlops\notebooks\catastrophes-climatiques-rag\catastrophes-climatiques-rag\notebooks\..\src\agents\tools.py:133: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


🔍 Résultats de recherche pour : « catastrophes climatiques 2024 bilan »

1. **Retour sur 2024 : des catastrophes climatiques en chaîne**
   https://www.lemonde.fr/planete/article/2025/01/09/retour-sur-2024-des-catastrophes-climatiques-en-chaine_6489984_3244.html
   Jan 9, 2025 · Il faut ajouter deux records au tableau d’une année exceptionnelle sur le plan climatique : 2024 devrait se classer comme la plus chaude jamais observée, devant 2023, et être la première pour...

2. **Les 10 plus grosses catastrophes climatiques de 2024 ont coûté au …**
   https://www.franceinfo.fr/replay-radio/le-billet-vert/les-10-plus-grosses-catastrophes-climatiques-de-2024-ont-coute-au-moins-200-milliards_6957365.html
   Dec 30, 2024 · Pourtant en 2024, année la plus chaude jamais enregistrée sur Terre, les températures de plus de 50 degrés qui ont touché l’Arabie saoudite, l’Inde ou la Thaïlande ont fait plus de 1 000 …

3. **Les catastrophes naturelles en France | Bilan 1982-2024**
   https://www.ccr.fr/

In [7]:
# Test 2 : information spécifique
result = web_search.invoke({'query': 'GIEC rapport AR6 conclusions principales', 'max_results': 3})
print(result)

c:\Users\Camille\mlops\notebooks\catastrophes-climatiques-rag\catastrophes-climatiques-rag\notebooks\..\src\agents\tools.py:133: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Aucun résultat trouvé pour : « GIEC rapport AR6 conclusions principales »


---
## 3. Test isolé — `calculator`

In [8]:
from src.agents.tools import calculator

tests_calcul = [
    '2 + 2',
    '15 * 1.08',                    # TVA 8%
    'sqrt(144)',                    # racine carrée
    '1000 * (1 + 0.02) ** 10',      # intérêts composés (projection CO2)
    'log10(1000000)',               # logarithme base 10
    '(45.5 - 32) * 5/9',           # Fahrenheit → Celsius
    '1 / 0',                       # division par zéro (cas d'erreur)
]

for expr in tests_calcul:
    result = calculator.invoke({'expression': expr})
    print(result)

🧮 2 + 2 = **4**
🧮 15 * 1.08 = **16.2**
🧮 sqrt(144) = **12**
🧮 1000 * (1 + 0.02) ** 10 = **1218.99**
🧮 log10(1000000) = **6**
🧮 (45.5 - 32) * 5/9 = **7.5**
❌ Erreur : division par zéro.


---
## 4. Test de l'agent LangGraph complet

L'agent choisit **automatiquement** le bon outil selon la question.

In [9]:
from src.agents.agent import run_agent

# Question → doit utiliser get_weather
q1 = 'Quelle est la météo actuelle à Lyon ?'
print(f'Question : {q1}')
print(run_agent(q1))
print('─' * 60)

Question : Quelle est la météo actuelle à Lyon ?
La météo actuelle à Lyon est caractérisée par un ciel dégagé avec une température de 21,3 °C (ressenti 19,4 °C), une humidité de 38 %, un vent de 6,9 km/h et des précipitations nulles. La pression atmosphérique est de 996,2 hPa. Ces informations sont fournies par l'API OpenMeteo, qui utilise le géocodage automatique pour obtenir les données météorologiques en temps réel pour n'importe quelle ville du monde.
────────────────────────────────────────────────────────────


In [10]:
# Question → doit utiliser calculator
q2 = 'Si les émissions de CO2 augmentent de 2% par an à partir de 37 Gt, '\
     'quel sera le niveau dans 20 ans ?'
print(f'Question : {q2}')
print(run_agent(q2))
print('─' * 60)

Question : Si les émissions de CO2 augmentent de 2% par an à partir de 37 Gt, quel sera le niveau dans 20 ans ?
Dans 20 ans, les émissions de CO2 seront de 54,98 Gt.
────────────────────────────────────────────────────────────


In [11]:
# Question → doit utiliser web_search
q3 = 'Quelles sont les dernières actualités sur les inondations en Europe en 2024 ?'
print(f'Question : {q3}')
print(run_agent(q3))
print('─' * 60)

Question : Quelles sont les dernières actualités sur les inondations en Europe en 2024 ?


c:\Users\Camille\mlops\notebooks\catastrophes-climatiques-rag\catastrophes-climatiques-rag\notebooks\..\src\agents\tools.py:133: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Les dernières actualités sur les inondations en Europe en 2024 sont disponibles sur des sites tels que Vigicrues.gouv.fr, Sudouest.fr, Franceinfo.fr, Bfmtv.com et Sentival.fr. Ces sources fournissent des informations sur les inondations, les crues, les dégâts et les vigilances météorologiques en France et dans le monde. Il est important de consulter ces sources pour rester informé sur la situation actuelle et les évolutions possibles.
────────────────────────────────────────────────────────────


In [12]:
# Question → réponse directe sans outil
q4 = 'Bonjour, comment tu t\'appelles ?'
print(f'Question : {q4}')
print(run_agent(q4))
print('─' * 60)

Question : Bonjour, comment tu t'appelles ?
Bonjour ! Je m'appelle Assistant Catastrophes Climatiques et Environnement. Je suis un assistant spécialisé dans les catastrophes climatiques et l'environnement, et je suis là pour vous aider à trouver des informations et des réponses à vos questions sur ces sujets. Je suis équipé de trois outils puissants : get_weather pour obtenir la météo actuelle, web_search pour effectuer des recherches sur le web et calculator pour effectuer des calculs mathématiques. N'hésitez pas à me poser vos questions !
────────────────────────────────────────────────────────────


In [13]:
# Question → combinaison météo + contexte climatique
q5 = 'Il fait combien de degrés à Marseille en ce moment ? '\
     'Est-ce normal pour la saison compte tenu du changement climatique ?'
print(f'Question : {q5}')
print(run_agent(q5))
print('─' * 60)

Question : Il fait combien de degrés à Marseille en ce moment ? Est-ce normal pour la saison compte tenu du changement climatique ?
La température actuelle à Marseille est de 17.2 °C. Pour savoir si cela est normal pour la saison compte tenu du changement climatique, il faudrait comparer cette température aux données historiques de la région. Le changement climatique peut entraîner des variations dans les températures moyennes et des événements climatiques extrêmes plus fréquents. Cependant, sans données spécifiques sur les tendances climatiques à Marseille, il est difficile de déterminer si cette température est anormale ou non.
────────────────────────────────────────────────────────────


---
## 5. Vérification de la structure du graphe LangGraph

In [14]:
from src.agents.agent import build_agent_graph

graph = build_agent_graph()
print('Nœuds du graphe :', list(graph.nodes.keys()))
print('✅ Graphe compilé avec succès')

Nœuds du graphe : ['__start__', 'call_model', 'tools']
✅ Graphe compilé avec succès


In [15]:
# Afficher le schéma ASCII du graphe
graph.get_graph().print_ascii()

        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
        +------------+        
        | call_model |        
        +------------+        
          .         *         
        ..           **       
       .               *      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 
